# Session 8 · Case study: first leaderboard submission (round L1)

*Author: course team · Licence: CC-BY-4.0*

Practice task of block 3: a logistic regression on the seven simple text features of Session 2, validated by time, written as `submission.csv` with the columns `review_id,label`.

Reference value on the public leaderboard: macro-F1 ≈ 0.49 (always "pos": 0.26). Rules: [case-study/README.md](../../../case-study/README.md).

Tasks for you are marked **Task**.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, f1_score
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

ROOT = Path.cwd()
while not (ROOT / "case-study").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DATA = ROOT / "case-study" / "data"

cols = ["review_id", "title", "text", "helpful_vote", "verified_purchase", "n_images", "date", "label"]
train = pd.read_parquet(DATA / "train.parquet", columns=cols).sort_values("date", ignore_index=True)
test = pd.read_parquet(DATA / "test.parquet")
print(train.shape, test.shape)
print(train["label"].value_counts(normalize=True).round(3).to_dict())

## 1 · The seven simple features

Fixed rules applied to each review separately: no fitting, so no leakage, and the same function serves training and test data.

In [ ]:
def simple_features(d):
    t = d["title"].fillna("") + " " + d["text"].fillna("")
    return pd.DataFrame({
        "log_len": np.log1p(t.str.len()),                     # length of title + text
        "n_excl": t.str.count("!"),                           # exclamation marks
        "n_quest": t.str.count(r"\?"),                        # question marks
        "n_neg": t.str.lower().str.count(r"\b(?:not|no|never|don't|didn't|doesn't|waste|return)\b"),
        "verified": d["verified_purchase"].astype(int),       # verified purchase
        "log_helpful": np.log1p(d["helpful_vote"]),           # helpful votes
        "n_images": d["n_images"]})                           # attached images

X_tr, y_tr = simple_features(train), train["label"]
X_te = simple_features(test)
X_tr.groupby(y_tr).mean().round(2)

## 2 · Validation that imitates the leaderboard

The leaderboard trains on the past and tests on the future, so we validate the same way: a `TimeSeriesSplit` over the training years, plus one out-of-time check on 2021 (train up to 2020).

`class_weight="balanced"` weights errors on the rare `neu` and `neg` classes more strongly during fitting. Without it, the model hardly ever predicts `neu`, which ruins macro-F1. Session 9 explains class weights in detail.

In [ ]:
candidates = {
    "always pos": DummyClassifier(strategy="constant", constant="pos"),
    "logistic regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "logistic regression, balanced": make_pipeline(StandardScaler(),
                                                   LogisticRegression(max_iter=1000, class_weight="balanced")),
}
tss = TimeSeriesSplit(5)
for name, m in candidates.items():
    s = cross_val_score(m, X_tr, y_tr, cv=tss, scoring="f1_macro")
    print(f"{name:32s} time-based CV macro-F1 {s.mean():.3f}  (folds {s.round(3)})")

In [ ]:
past, last = train["date"] < "2021-01-01", train["date"] >= "2021-01-01"
l1 = candidates["logistic regression, balanced"].fit(X_tr[past], y_tr[past])
pred_2021 = l1.predict(X_tr[last])
print("out-of-time macro-F1 (2021):", round(f1_score(y_tr[last], pred_2021, average="macro"), 3))
print(classification_report(y_tr[last], pred_2021, digits=3))
ConfusionMatrixDisplay.from_predictions(y_tr[last], pred_2021, labels=["neg", "neu", "pos"], cmap="Blues",
                                        colorbar=False)
plt.title("L1 model, reviews of 2021 (trained up to 2020)")
plt.show()

**Task 1.** Read the confusion matrix. Which class is confused with which? Why is the F1 of `neu` so low, and why can seven counts of punctuation and length not separate a 3-star review from a 4-star review?

## 3 · Fit on all training data and write the submission

In [ ]:
l1 = candidates["logistic regression, balanced"].fit(X_tr, y_tr)
submission = pd.DataFrame({"review_id": test["review_id"], "label": l1.predict(X_te)})

# checks before uploading
assert list(submission.columns) == ["review_id", "label"]
assert len(submission) == len(test) and submission["review_id"].is_unique
assert submission["label"].isin(["neg", "neu", "pos"]).all()

out = ROOT / "submission.csv"
submission.to_csv(out, index=False)
print("wrote", out, len(submission), "rows")
print(submission["label"].value_counts(normalize=True).round(3).to_dict())

Upload `submission.csv` (in the repository root) to the course leaderboard. Only the lecturer has the hidden labels (`case-study/data/instructor/solution.csv`) and can score locally from the repository root:

```bash
uv run --with pandas --with scikit-learn python case-study/score.py submission.csv
```

The next cell does the same if the solution file is present (lecturer only).

In [ ]:
solution = DATA / "instructor" / "solution.csv"
if solution.exists():
    sol = pd.read_csv(solution).merge(submission, on="review_id", suffixes=("", "_pred"))
    for usage, part in sol.groupby("Usage"):
        print(f"{usage:8s} macro-F1 {f1_score(part['label'], part['label_pred'], average='macro'):.3f}")
else:
    print("No solution file: submit to the leaderboard to get your score.")

**Task 2.** Record in your team log: features, model, time-based CV macro-F1, 2021 out-of-time macro-F1 and, once available, the public leaderboard score. Are they consistent?

**Task 3 (optional).** Add one simple feature of your own (for example the number of capital letters, or a count of words like "great", "love", "terrible"). Compute it with a fixed rule, re-run the time-based CV and keep it only if the score improves by more than the fold-to-fold spread.

> **Do not** add `train_avg_rating` from the product table: for training reviews it includes the review's own rating (target leakage; see Session 7, theory page 03).